In [ ]:
import os
import runpy
import subprocess
from base64 import b64encode
from pathlib import Path

from google.colab import drive, userdata

DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/PASTE_FOLDER_ID"
# Add the linked folder to My Drive as a shortcut named DD2610-PartA.
drive_folder = Path("/content/drive/MyDrive/DD2610-PartA")
repo_url = "https://github.com/gorrast/DD2610-Final-Project.git"
checkout = Path("/content/DD2610-partA")

drive.mount("/content/drive")
if not drive_folder.is_dir():
    raise FileNotFoundError(f"Add a My Drive shortcut named DD2610-PartA to {DRIVE_FOLDER_URL}.")

def git(*args):
    token = userdata.get("GITHUB_TOKEN")
    auth = b64encode(f"x-access-token:{token}".encode()).decode()
    subprocess.run(
        ["git", *args],
        env=dict(
            os.environ,
            GIT_CONFIG_COUNT="1",
            GIT_CONFIG_KEY_0=f"http.{repo_url}.extraHeader",
            GIT_CONFIG_VALUE_0=f"Authorization: Basic {auth}",
            GIT_TERMINAL_PROMPT="0",
        ),
        check=True,
    )

if checkout.exists():
    branch = subprocess.check_output(
        ["git", "-C", str(checkout), "branch", "--show-current"], text=True
    ).strip()
    if branch != "partA":
        raise RuntimeError(f"Expected the partA branch in {checkout}.")
    git("-C", str(checkout), "pull", "--ff-only", repo_url, "partA")
else:
    git("clone", "--depth", "1", "--branch", "partA", repo_url, str(checkout))

# Set this secret to an approved direct ImageNet URL for a team-restricted folder only.
try:
    imagenet_val_url = userdata.get("IMAGENET_VAL_URL").strip()
except userdata.SecretNotFoundError:
    imagenet_val_url = None

helper = runpy.run_path(str(checkout / "download_datasets.py"))
helper["download_datasets"](drive_folder, imagenet_val_url)